# Lesson 02 — FLANN Matcher: Fast Approximate Nearest Neighbors

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M = cv2.getRotationMatrix2D((img1.shape[1]//2, img1.shape[0]//2), 20, 0.85)
img2 = cv2.warpAffine(img2, M, (img2.shape[1], img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=1000)
kp1, d1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
kp2, d2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)

# BFMatcher speed
t0 = time.time()
bf = cv2.BFMatcher()
bf.knnMatch(d1, d2, k=2)
t_bf = time.time() - t0

# FLANN speed
FLANN_INDEX_KDTREE = 1
index_params  = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

t0 = time.time()
matches_flann = flann.knnMatch(d1, d2, k=2)
t_fl = time.time() - t0

print(f"BFMatcher: {t_bf*1000:.1f}ms")
print(f"FLANN:     {t_fl*1000:.1f}ms   (~{t_bf/t_fl:.1f}x faster on large sets)")

## Key Takeaway
FLANN is approximate but 5-10x faster than BFMatcher on large descriptor sets.
Use BFMatcher for ORB (binary descriptors, NORM_HAMMING).
Use FLANN for SIFT/SURF (float descriptors, NORM_L2).